<a href="https://colab.research.google.com/github/nicolastibata/MINE_4210_ADL_202620/blob/main/assingments/assingment_4/MINE_4210_ADL_202620_T4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo ADL](https://github.com/nicolastibata/MINE_4210_ADL_202620/blob/main/docs/logo.png?raw=true)


# **Taller 4: Modelos generativos para imágenes de células sanguíneas**

### **Contexto y Objetivos**

En el laboratorio clínico el recuento diferencial de células en un frotis de sangre periférica orienta el diagnóstico de infecciones, anemias y enfermedades hematológicas. Hacerlo manualmente toma tiempo y sus resultados varían entre observadores, por lo que muchos laboratorios utilizan clasificadores automáticos de imágenes.

Un laboratorio de hematología de un hospital quiere entrenar su propio clasificador, pero tiene solo 200 imágenes de basófilos, una célula poco frecuente en las muestras. Un miembro del equipo propone usar modelos generativos para producir imágenes sintéticas de basófilos, con el fin de complementar los datos de entrenamiento. Esto permitiría, además, compartir imágenes sintéticas en lugar de las reales con otras instituciones, para proteger la información de los pacientes. Antes de decidir, se quiere hacer una evaluación técnica que permita responder a tres inquietudes: ¿las imágenes generadas son realistas y variadas?, ¿el modelo reproduce imágenes reales de pacientes? ¿las imágenes sintéticas mejoran el clasificador?

En este taller compararán un VAE y un modelo de difusión para generar imágenes de basófilos, y evaluarán si las imágenes sintéticas mejoran un clasificador.

Conjunto de datos. Se utilizará BloodMNIST  (https://medmnist.com/). Este dataset está compuesto por 17.092 imágenes de células de sangre periférica, en color y de 28×28 píxeles, clasificadas en 8 tipos (0: basófilo, 1: eosinófilo, 2: eritroblasto, 3: granulocito inmaduro, 4: linfocito, 5: monocito, 6: neutrófilo, 7: plaqueta). Las imágenes originales se adquirieron en el Hospital Clínic de Barcelona, a partir de personas sin infecciones, enfermedades hematológicas u oncológicas, ni tratamiento farmacológico.

Debido al protocolo de recolección de las imágenes, los basófilos representan cerca del 7 % del conjunto, una proporción superior a la práctica clínica. Para simular la situación real del laboratorio, entrenarán los generadores únicamente con 200 basófilos de la partición oficial de entrenamiento. Seleccionen el subconjunto de la siguiente forma:

```python
import numpy as np
from medmnist import BloodMNIST

train = BloodMNIST(split="train", download=True)
val   = BloodMNIST(split="val",   download=True)
test  = BloodMNIST(split="test",  download=True)

idx_baso = np.where(train.labels.ravel() == 0)[0]
rng = np.random.default_rng(42)
idx_sub = rng.choice(idx_baso, size=200, replace=False)
basofilos_train = train.imgs[idx_sub]   # (200, 28, 28, 3)
```

Llamaremos N al número total de basófilos en la partición de entrenamiento. El subconjunto de 200 forma parte de ellos. Para los modelos generativos, normalicen los píxeles al rango [−1, 1]. El conjunto se usa con fines académicos, y los resultados no deben interpretarse como evidencia sobre la utilidad clínica de los modelos. Con base en el tipo de licencia de este conjunto de imágenes deben hacer la citación que corresponde.

<br>

**Actividades**

**1. Exploración y clasificador de referencia.**

- Revisen la distribución de clases en entrenamiento, validación y prueba, y reporten N.
- Muestren seis imágenes reales por clase. ¿Qué distingue visualmente a los basófilos y con cuáles podrían confundirse?
- Construyan un conjunto de entrenamiento que contenga únicamente los 200 basófilos seleccionados y todas las imágenes de entrenamiento de las otras siete clases. Entrenen una CNN con este conjunto, utilizando la siguiente arquitectura:
  ```python
  Conv2D(32, 3×3, ReLU) + MaxPooling(2×2)
  Conv2D(64, 3×3, ReLU) + MaxPooling(2×2)
  Flatten + Dense(128, ReLU) + Dropout(0.3)
  Dense(8, softmax)
  ```
Utilicen Adam (learning rate = 1e-3), máximo 10 épocas y EarlyStopping sobre val_loss (patience=3, restaurando los mejores pesos). Reporten, en el conjunto de test, el F1 macro, y el F1_score y el recall de la clase basófilo.

Conserven el clasificador entrenado y sus resultados, ya que se utilizarán como referencia para evaluar las imágenes generadas y determinar, en la actividad 4, si el aumento sintético mejora la clasificación.

<br>

**2. Generación de imágenes con VAE y DDPM.**

Entrenen ambos modelos exclusivamente con los mismos 200 basófilos. Pueden partir de los ejemplos del laboratorio demostrativo (Laboratorio 10).

**2.1. Modelo 1. VAE**

- Encoder y decoder convolucionales (28 → 14 → 7 y reconstrucción hasta 28 × 28), adaptados a tres canales RGB.
- Espacio latente de dimensión 16; pérdida de reconstrucción más divergencia KL. Documenten su implementación.
- Máximo 300 épocas o 20 minutos de GPU, lo que ocurra primero. Presenten la evolución de las pérdidas.

**2.2 Modelo 2. Modelo de difusión (DDPM)**

Implementen un modelo de difusión utilizando la librería diffusers de Hugging Face. Utilicen la siguiente configuración:

- Arquitectura: UNet2DModel, adaptada a imágenes RGB de 28 × 28 píxeles.
- Proceso de difusión: DDPMScheduler con 1000 pasos de entrenamiento.
- Función de pérdida: error cuadrático medio entre el ruido añadido a las imágenes y el ruido predicho por la U-Net.
- Optimizador: AdamW, con tasa de aprendizaje 1e-4 y tamaño de lote máximo de 64.
- Entrenamiento: máximo de 60 minutos de GPU.

Registren la evolución de la pérdida y documenten la arquitectura utilizada, el número de actualizaciones de los parámetros y el tiempo de entrenamiento.

No se exige alcanzar una calidad visual específica. Si las imágenes tienen defectos, analícenlos y discútanlos. Los 1000 pasos del scheduler describen el proceso de difusión y no equivalen a 1000 actualizaciones de parámetros.

**2.3 Generación**

Generen 300 imágenes sintéticas con cada modelo. Para DDPM indiquen el número de pasos de inferencia utilizado. Conserven los modelos y las imágenes para las actividades siguientes.

<br>

**3. Evaluación de las imágenes generadas**.

| Aspecto | Evaluación |
|-|-|
| Calidad visual y consistencia con la clase | Muestren 16 imágenes por generador y 16 basófilos reales del conjunto de test. Calculen el porcentaje de las 300 sintéticas clasificadas como basófilos por la CNN <br> y compárenlo con su recall sobre basófilos reales de test. | Distribución de longitudes en tokens, distribución de clases y análisis de longitud por ODS. |
| Diversidad | Distancia euclidiana promedio entre pares de 100 imágenes sintéticas por modelo y 100 basófilos reales de test, complementada con inspección visual. |
| Posible memorización | Para cada imagen sintética, busquen el vecino más cercano entre los 200 basófilos de entrenamiento. Muestren los cinco pares más similares por modelo <br> y comparen sus distancias con las de basófilos reales de test frente a esos mismos 200 ejemplos. |

Antes de calcular distancias, lleven las imágenes reales y las sintéticas a la misma escala de valores (por ejemplo, [0, 1]). Con base en estos resultados, elijan el generador que usarán para el aumento sintético y justifiquen la decisión.

<br>

**4. Evaluaciones en el clasificador**. En la actividad 1 entrenaron una CNN utilizando 200 basófilos reales y todas las imágenes de las otras siete clases. Ahora evaluarán si incorporar imágenes sintéticas permite mejorar su desempeño. Para ello:

- Para la construcción de un clasificador con sobre muestreo. Preparen un conjunto de entrenamiento con todas las imágenes de las otras siete clases utilizadas en la actividad 1 y los 200 basófilos reales repetidos hasta completar N (por ejemplo, con np.resize(idx_sub, N)).
- Para la construcción de un clasificador con aumento sintético. Preparen un conjunto de entrenamiento con los 200 basófilos reales, todas las imágenes de las otras siete clases y (N − 200) imágenes sintéticas producidas por el generador seleccionado en la actividad 3, hasta completar N basófilos. Si necesitan más de las 300 imágenes generadas inicialmente, produzcan las adicionales con el mismo modelo. Apliquen a las imágenes sintéticas el mismo formato y preprocesamiento utilizado para entrenar la CNN.
- Entrenen una CNN con cada conjunto, utilizando la misma arquitectura, configuración de entrenamiento y semilla de la actividad 1.
- Evalúen ambas CNN sobre la misma partición oficial de test utilizada en la actividad 1. Presenten en una tabla el F1 macro, y el F1_score y el recall de la clase basófilo de los tres clasificadores (referencia, sobre muestreo y aumento sintético), junto con sus matrices de confusión.

<br>

**5. Análisis**. A partir de los resultados obtenidos, respondan las siguientes preguntas:

1. Comparación de los modelos generativos. ¿Qué diferencias encuentran entre el VAE y el DDPM en términos de calidad visual, diversidad, posible memorización  y costo de generación? ¿Cómo se relacionan estas diferencias con el funcionamiento de ambos modelos? ¿Qué limitaciones tienen las métricas utilizadas para evaluar las imágenes, considerando además que la CNN evaluadora se entrenó con los mismos 200 basófilos que usaron los generadores?
2. Utilidad de las imágenes sintéticas. ¿La incorporación de imágenes sintéticas mejoró el desempeño del clasificador frente al entrenado únicamente con imágenes reales y, sobre todo, frente al entrenado con sobre muestreo? ¿Qué relación encuentran entre los resultados de clasificación y la calidad y diversidad de las imágenes generadas? Dado que cada clasificador se entrenó con una sola semilla, ¿qué tan confiables son las diferencias observadas? ¿Qué limitaciones tiene este experimento para determinar si las imágenes sintéticas aportan información nueva?
3. Recomendación al laboratorio. En una nota de máximo 300 palabras, indiquen si recomiendan utilizar imágenes sintéticas para complementar los datos de entrenamiento y si consideran adecuado compartirlas con otras instituciones en lugar de las imágenes reales. Justifiquen sus recomendaciones con base en los resultados obtenidos y señalen las limitaciones y la evidencia adicional que sería necesaria antes de aplicar estas soluciones en un entorno clínico.

**Bono (hasta 15 % adicional).** A partir del código del laboratorio, entrenen una GAN con los mismos 200 basófilos (Adam, learning rate 2e-4, betas=(0.5, 0.999), máximo 20 minutos de GPU). Generen 300 imágenes y comparen calidad visual, consistencia con la clase y diversidad con VAE y DDPM mediante los procedimientos correspondientes de la actividad 3. ¿Hay indicios de colapso de modos? ¿Qué evidencia los respalda? ¿Cambiarían la elección del generador?

### **Entregable**

Notebook (.ipynb) con todas las celdas ejecutadas, organizado por actividades, que incluya código, tablas, visualizaciones, análisis, nota al laboratorio,  referencias a recursos externos y con todas las indicaciones anteriormente estipuladas en la sección de laboratorios en **Bloque Neón**. Documenten decisiones y permitan reproducir los experimentos en la medida de lo posible.



### **Rúbrica**

| Criterio | Peso | Qué se evalúa |
|-|-|-|
| 1. Exploración y clasificador de referencia | 15 % | Exploración y distribución de clases, selección de los 200 basófilos, entrenamiento de la CNN y evaluación mediante las métricas solicitadas. |
| 2. Generación de imágenes con VAE y DDPM | 30 % | Implementación y entrenamiento de ambos modelos, documentación de configuraciones y pérdidas, y generación de 300 imágenes por modelo con su tiempo de generación. |
| 3. Evaluación de las imágenes generadas | 20 % | Evaluación comparativa de calidad visual, consistencia con la clase, diversidad y posible memorización; interpretación y selección justificada del generador. |
| 4. Evaluación en el clasificador | 15 % | Preparación de los conjuntos con sobre muestreo y con aumento sintético, entrenamiento de las dos CNN y comparación de su desempeño con el clasificador de referencia. |
| 5. Análisis | 20 % | Comparación de los generadores, interpretación de la utilidad del aumento sintético frente al sobre muestreo, discusión de limitaciones y recomendación sustentada al laboratorio. |
| 6. Bono. GAN | +15 % | Entrenamiento de la GAN, generación y evaluación de imágenes, comparación con VAE y DDPM, análisis de posible colapso de modos y justificación de la elección del generador. |

> *Uso de IAG en la elaboración del enunciado de este laboratorio.
El equipo docente declara de forma explícita que, en el contexto de este laboratorio, la IAG fue utilizada como apoyo en la búsqueda del conjunto de datos y en la redacción del enunciado. Todo el contenido fue revisado, verificado y ajustado por el equipo docente.*
